# CURATOR on Kaggle GPUs
Settings: **GPU T4 x2** (or P100), **Internet On**, the `curator-gpu-kaggle` dataset attached.
Recommended: a Kaggle secret `HF_TOKEN` (a Hugging Face write token), so every run is mirrored while it trains and resumes after a lost session. See README_KAGGLE.md in the zip.

In [ ]:
%%bash
# 1) unpack the bundle
mkdir -p /kaggle/working/curator && cd /kaggle/working/curator
unzip -q -o /kaggle/input/*/curator-gpu-kaggle.zip
ls
nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
%cd /kaggle/working/curator
# 2) install the stack for this GPU (T4: vLLM virtualenv; P100: HF backend)
!python run.py setup

In [ ]:
# where runs are mirrored while they train (needs the HF_TOKEN secret); '' keeps them local only
SYNC = 'hf://YOUR_HF_USERNAME/curator-runs'
SYNC_ARG = f'--sync-to {SYNC}' if SYNC and 'YOUR_HF_USERNAME' not in SYNC else ''
print(SYNC_ARG or 'no sync target: Save Version before closing the session')

In [ ]:
# 3) ~10 min end-to-end check: 3 rounds + calibration + checkpoint + restore from the synced copy + resume
!python run.py smoke {SYNC_ARG}

In [ ]:
# 4) the comparison (T4 x2: two runs in parallel). Rerunning this cell in a new session RESUMES.
# On P100 prefer: !python run.py sysid --calib-items 50 --final-items 100 {SYNC_ARG}
!python run.py compare --methods curator,uniform,lp --seeds 0 {SYNC_ARG}

In [ ]:
# where every run is (safe to run any time)
!python run.py status

In [ ]:
# 5) results (copy these back)
print(open('/kaggle/working/curator_runs/comparison.md').read())

Without a sync target: **Save Version before closing**, then in the next session attach this version's output as an input and rerun step 4 with `--resume-from /kaggle/input/<output>/curator_runs`.